# **Trabajo Practico N° 2**
**Nuñez Juliana, Segovia Lucas**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [2]:
df_diabetes = pd.read_csv("./data/diabetes.csv")
df_diabetes.head()

from sklearn.model_selection import train_test_split
df_diabetes = df_diabetes.drop(columns=["Unnamed: 0"])

train_set, test_set = train_test_split(df_diabetes, test_size=0.2, stratify=df_diabetes["Outcome"], random_state=42)

## **Actividad 1**

¿Sería posible utilizar esta misma estrategia dentro de un pipeline que deba procesar pacientes nuevos?
No es posible. Como ya se explico en la entrega anterior, estrategia de imputación por clase requiere agrupar los datos utilizando el diagnóstico de las pacientes (Outcome) y no se puede depender de esta etiqueta para completar los datos, ya que provocaría una fuga de información que invalida el modelo.  

¿Qué información estará disponible cuando el modelo se utilice para realizar predicciones y cuál no?
Cuando el modelo se utilice para evaluar pacientes nuevos, solo estarán disponibles las variables clínicas predictoras (Glucose, BloodPressure, BMI, etc). La variable objetivo (Outcome) no estará disponible, ya que es lo que buscamos predecir.  

Teniendo en cuanta lo dicho se concluye que la estrategia de división por grupos no es adecuada, por lo que se presenta la siguiente alternativa:

Aplicar una imputación global utilizando la mediana general calculada de manera unificada sobre todo el conjunto de entrenamiento, omitiendo completamente la variable Outcome durante el cálculo (.drop("Outcome", axis=1)), utilizando la clase SimpleImputer(strategy="median"). Es la única forma de garantizar consistencia metodológica entre la fase de entrenamiento y las futuras predicciones, ya que crea una regla matemática que depende de las variables de entrada. También se tiene en cuanta que se elige la mediana  porque los histogramas del análisis exploratorio demostraron que atributos como Age, Pregnancies e Insulin presentan una fuerte distribución sesgada hacia la izquierda, entonces la mediana es un descriptor estadístico mucho más preciso y robusto.

In [3]:
diabetes_train_X = train_set.drop("Outcome", axis=1)
columnas_con_cero= ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

df_global= diabetes_train_X.copy()
df_global[columnas_con_cero]=df_global[columnas_con_cero].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
df_global[columnas_con_cero] = imputer.fit_transform(df_global[columnas_con_cero])

train_set[columnas_con_cero] = train_set[columnas_con_cero].replace(0, np.nan)
medianas_globales = train_set[columnas_con_cero].median()
print(medianas_globales)

Glucose          117.0
BloodPressure     72.0
SkinThickness     29.0
Insulin          125.0
BMI               32.4
dtype: float64


# **Actividad 2** 

Cuadro estadístico auxiliar:

In [4]:
df_diabetes.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,767.000000,756.000000,763.000000,767.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.195567,20.862434,80.322412,32.034289,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.206609,15.865314,115.439459,7.804050,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,63.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,36.000000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,128.500000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


**División del dataset:** Se separan los datos usando un test_size=0.2, random_state=42 y el parámetro stratify=df["Outcome"]. 

El análisis realizado de los estadísticos demostró que las clases están desbalanceadas (aprox. 65% de personas sin diabetes, 35% con diabetes). La estratificación garantiza que los conjuntos mantengan la proporción original, evitando sesgos en la evaluación. Realizar la partición antes del preprocesamiento es vital para evitar la fuga de información, asegurando que el modelo no aprenda de los datos de prueba.  

**Tratamiento de valores inválidos:** Se reemplazan los ceros por valores nulos (NaN) únicamente en las variables Glucose, BloodPressure, SkinThickness, Insulin y BMI. 

En los parámetros clínicos mencionados un valor de 0 es fisiológicamente imposible y representa un dato no registrado o un error de medición, los cálculos estadísticos realizados por los algoritmos se distorsionarían. En variables como Pregnancies, el 0 es un dato real, biológicamente válido y no debe modificarse.

**Imputación de valores faltantes:** a las columnas con ceros reemplazados por NaN se les aplica una imputación utilizando la mediana del conjunto de entrenamiento (SimpleImputer(strategy="median")). 

Al observar los estadísticos se notó que la media y la mediana difieren, lo que indica asimetría. La mediana es un estadístico mas representativo frente a valores atípicos y asimetrías. Se usa una imputación global para mantener un flujo de trabajo ciego e independiente de la etiqueta final. 

**Transformación de atributos con distribuciones fuertemente sesgadas:** se aplica una transformación logarítmica (FunctionTransformer(np.log1p)) exclusivamente a la variable Insulin. 

El análisis exploratorio evidenció que la insulina presenta una fuerte distribución sesgada, acumulando la mayoría de los valores en números bajos, los cuales se disparan en el ultimo cuartil. Aplicar el logaritmo comprime estos rangos altos y expande los bajos, acercando la distribución a una forma de campana normal.  

**Escalado de atributos numéricos:** a todas las variables se les aplica una estandarización mediante StandardScaler(). Los atributos trabajan en escalas completamente distintas. Estandarizar fuerza a que todas las variables tengan media 0 y desviación estándar 1, evitando que las variables con números más grandes dominen el peso de los cálculos matemáticos del algoritmo.

In [5]:
insulin_attribs = ["Insulin"]
otros_con_cero_attribs = ["Glucose", "BloodPressure", "SkinThickness", "BMI"]
sin_cero_attribs = ["Pregnancies", "DiabetesPedigreeFunction", "Age"]

# **Actividad 3** 

**Funcion Auxiliar**

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import numpy as np

def zero_to_nan(X):
    """Reemplaza ceros por NaN, para columnas donde 0 no es fisiológicamente válido."""
    return np.where(X == 0, np.nan, X)

**Pipelines**

In [7]:
pipeline_insulin = make_pipeline(
    FunctionTransformer(zero_to_nan, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log1p, inverse_func=np.expm1, feature_names_out="one-to-one"),
    StandardScaler(),
)

pipeline_otras_con_ceros = make_pipeline(
    FunctionTransformer(zero_to_nan, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)

pipeline_sin_ceros = make_pipeline(
    StandardScaler(),
)

**ColumnTransformer**

In [8]:
preprocesador = ColumnTransformer([
    ("insulin", pipeline_insulin, insulin_attribs),
    ("otras_con_ceros", pipeline_otras_con_ceros, otros_con_cero_attribs),
    ("sin_ceros", pipeline_sin_ceros, sin_cero_attribs),
])

preprocesador  # muestra el diagrama

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('insulin', ...), ('otras_con_ceros', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name``

**Split**

In [9]:
X_train = train_set.drop("Outcome", axis=1)
y_train = train_set["Outcome"]

X_test = test_set.drop("Outcome", axis=1)
y_test = test_set["Outcome"]

X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

# **Actividad 4**

**¿Cuál es la diferencia entre fit(), transform() y fit_transform()?**
*  fit(): Aprende los parámetros necesarios de la transformación a partir de los datos (ej.: Mediana para imputar, media/desvio para escalar).
* transform(): Aplica una transformación ya aprendida (mediante fit) sobre un conjunto de datos, sin hacer calculos sobre nuevos parametros.
* fit_transform(): Combina ambos pasos en uno, aprende los parametros y aplica la transformación sobre el mismo conjunto de datos, en un solo llamado.


**¿Qué método debe utilizarse sobre el conjunto de entrenamiento?**

Se debe utilizar fit_transform(), porque es el conjunto a partir del cual se deben aprender los parámetros que luego se utilizaran en todo el proceso posterior, incluyendo tests y datos futuros.


**¿Qué método debe utilizarse sobre el conjunto de prueba?**

Solo se debe usar transform(). No se pueden usar ni fit() ni fit_transform() sobre el test.


**¿Por qué aplicar fit() sobre el conjunto de prueba sería un error metodológico?**

Porque aplicar fit() implica recalcular parametros (medians, medias, desvios) usando informacion del conjunto de prueba. Esto filtra la informacion y el modelo terminaria siendo entrenado/preparado con informacion que no deberia poseer para ser evaluada en cuanto a su desempeño en un escenario real frente a datos nuevos. Como resultado, tendriamos metricas artificialmente optimistas y no representativas del comportamiento real del modelo, esto es la fuga de informacion discutida en el TP1.


In [10]:
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

In [11]:
X_train_prep[:2].round(3)

array([[-2.108, -1.056, -0.827, -1.918, -0.77 , -0.851,  0.311, -0.792],
       [-3.437,  0.144,  0.478, -0.23 , -0.418,  0.357, -0.116,  0.561]])

# **Actividad 5:**

### **Importancia de la verificación:**

**Verificar la ausencia de valores faltantes:** los modelos estándar de aprendizaje automático en Scikit-Learn requieren matrices de datos numéricas completas. Si el pipeline falló y dejó pasar un valor NaN el algoritmo no podrá realizar los cálculos y el entrenamiento fallará.

Usamos np.isnan(), porque la salida es un arreglo de nunpy. Funciona como un escáner, pasa por todos los datos de tu tabla y devolverte una matriz idéntica en tamaño, pero llena de valores booleanos (True si es NaN, False si es un número válido), luego sum los cuenta, de estar ambos conjuntos preparados correctamente, deberiamos ver 0 valores faltantes.

**Verificar la igualdad de atributos:** el modelo ajusta sus parámetros y pesos matemáticos basándose en la estructura exacta de la matriz con la que fue entrenado (en este caso, 8 columnas). Si el conjunto de prueba o cualquier dato futuro tuviera un atributo de más o de menos, o un orden distinto, el modelo sufriría un error de dimensionalidad y sería incapaz de predecir.

.shape[1]: devuelve las cantidad de columnas de los conjuntos de entrenamiento y de testeo

**Obtener los nombres de las variables:** como se usó un ColumnTransformer, los datos originales fueron separados en ramas, reordenados y procesados. El transformador les asigna nuevos nombres combinando el nombre de la rama (otras_con_ceros, insulin, sin ceros) y la columna (nombre original, Glucose, BMI, SkinThickness, etc). Recuperar este mapeo es fundamental para la interpretabilidad del modelo en el futuro para saber a qué característica corresponde cada coeficiente o nivel de importancia asignado por el algoritmo.

In [12]:
# Verificar que no existan valores faltantes en los datos procesados
nulos_train = np.isnan(X_train_prep).sum()
nulos_test = np.isnan(X_test_prep).sum()

print(f"Valores faltantes en entrenamiento: {nulos_train}")
print(f"Valores faltantes en prueba: {nulos_test}")

# Que entrenamiento y prueba posean los mismos atributos
columnas_train = X_train_prep.shape[1]
columnas_test = X_test_prep.shape[1]

print(f"\nCantidad de columnas del conjunto de entrenamiento: {columnas_train}")
print(f"Cantidad de columnas del conjunto de prueba: {columnas_test}")
print(f"¿Poseen exactamente los mismos atributos (columnas)?: {columnas_train == columnas_test}")

# Obtenga los nombres de las variables generadas por el pipeline
nombres_variables = preprocesador.get_feature_names_out()
print("\nNombres de las variables generadas por el pipeline:")
for nombre in nombres_variables:
    print(f"- {nombre}")

Valores faltantes en entrenamiento: 0
Valores faltantes en prueba: 0

Cantidad de columnas del conjunto de entrenamiento: 8
Cantidad de columnas del conjunto de prueba: 8
¿Poseen exactamente los mismos atributos (columnas)?: True

Nombres de las variables generadas por el pipeline:
- insulin__Insulin
- otras_con_ceros__Glucose
- otras_con_ceros__BloodPressure
- otras_con_ceros__SkinThickness
- otras_con_ceros__BMI
- sin_ceros__Pregnancies
- sin_ceros__DiabetesPedigreeFunction
- sin_ceros__Age
